# 01 · Preprocessing — from raw data to a `DolceSet`

Every model eats a **`DolceSet`**: a container holding feature matrix `X`, target `y`, and a record of applied transformations. Steps:

1. **Load** a table, name the target.
2. **Inspect** it.
3. **Encode** categoricals into numbers.
4. **Scale** features to comparable ranges.
5. **Encode** class labels, for classifiers.

## 1 · Loading data

`DolceSet` loads from a dict (`load_from_dict`) or a Polars DataFrame (`load_from_polars_dataframe`). Pass `target_col`; it separates features from target.

Toy dataset: apartment **price** (target) from **size**, **rooms**, and categorical **district**.

In [1]:
from dolcestat.preprocessing import DolceSet

raw = {
    "size_m2":  [55, 80, 120, 42, 95, 150, 60, 200, 70, 110],
    "rooms":    [2, 3, 4, 1, 3, 5, 2, 6, 2, 4],
    "district": ["center", "suburb", "center", "suburb", "center",
                 "suburb", "center", "suburb", "center", "suburb"],
    "price_k":  [180, 210, 340, 120, 260, 400, 175, 520, 190, 300],
}

data = DolceSet()
data.load_from_dict(raw, target_col="price_k")

## 2 · Inspecting the set

- `get_features()` / `get_target()` — feature names / target name
- `get_colnames()` — every column, target included
- `get_column(name)` — one column as a NumPy array
- `.X` / `.y` — the feature matrix and target vector models use

In [2]:
print("features:", data.get_features())
print("target:  ", data.get_target())
print("X shape: ", data.X.shape, " y shape:", data.y.shape)
data.get_column("size_m2")

features: ['size_m2', 'rooms', 'district']
target:   price_k
X shape:  (10, 3)  y shape: (10,)


array([ 55,  80, 120,  42,  95, 150,  60, 200,  70, 110])

## 3 · Encoding categoricals

`one_hot_encode` turns a categorical column into 0/1 indicator columns, one per category, using **drop-first** encoding (*k* categories → *k − 1* columns). This avoids the dummy-variable trap (perfectly collinear indicators).

The target `price_k` stays out of the feature list.

In [3]:
print("before:", data.get_features())
data.one_hot_encode("district")
print("after: ", data.get_features())

before: ['size_m2', 'rooms', 'district']
after:  ['size_m2', 'rooms', 'district_suburb']


## 4 · Scaling

`size_m2` runs into the hundreds, `rooms` stays single-digit. Distance-based models (KNN) and gradient-based optimizers need comparable feature ranges. Two strategies:

- **`"standardize"`** — subtract mean, divide by std (→ mean 0, std 1).
- **`"min-max"`** — rescale to `[0, 1]`.

`scale(strategy)` transforms every feature; `scale(strategy, "column")` targets one. `scaling_info` records what was applied.

In [4]:
data.scale("standardize", "size_m2")
data.scale("min-max", "rooms")

for step in data.scaling_info["details"]:
    print(step["strategy"], "→", step["columns"])

data.X[:3]

standardize → ['size_m2']
min-max → ['rooms']


array([[-0.88838837,  0.2       ,  0.        ],
       [-0.37427473,  0.4       ,  1.        ],
       [ 0.44830709,  0.6       ,  0.        ]])

### Applying training statistics to new data

To avoid **leakage**, scale test data with *training* statistics, not its own. Pass the recorded stats back via `mean`/`std` (or `min`/`max`):

In [5]:
train = DolceSet()
train.load_from_dict({"x": [1.0, 2.0, 3.0, 4.0], "y": [0, 0, 1, 1]}, target_col="y")
train.scale("standardize", "x")
stats = train.scaling_info["details"][0]["stats"]

test = DolceSet()
test.load_from_dict({"x": [5.0, 6.0], "y": [1, 1]}, target_col="y")
test.scale("standardize", "x", mean=stats["mean"], std=stats["std"])
test.X.flatten()

array([1.93649167, 2.71108834])

## 5 · Encoding class labels

Classifiers need the class labels as the integers `0, 1, …, K-1`. For any other labelling — text, or numbers that start at 1 — `encode_target()` numbers the sorted distinct values and keeps the originals in `classes_` (`decode_target` maps integers back).

Encode a test set with the **training** mapping, by passing it back in; a label the training set never saw is an error.

In [6]:
flowers = DolceSet()
flowers.load_from_dict(
    {"petal": [1.2, 4.5, 1.4, 5.1, 4.0, 1.3],
     "kind": ["setosa", "versicolor", "setosa", "virginica", "versicolor", "setosa"]},
    target_col="kind",
)
flowers.encode_target()
print("encoded target:", flowers.y, "  classes_:", flowers.classes_)

new_flowers = DolceSet()
new_flowers.load_from_dict({"petal": [4.2, 1.1], "kind": ["versicolor", "setosa"]}, target_col="kind")
new_flowers.encode_target(classes=flowers.classes_)   # same mapping as the training set
print("test target:   ", new_flowers.y, "->", new_flowers.decode_target(new_flowers.y))

encoded target: [0 1 0 2 1 0]   classes_: ['setosa' 'versicolor' 'virginica']
test target:    [1 0] -> ['versicolor' 'setosa']


## Recap

The `DolceSet` is loaded, encoded, and scaled. Next: [`02_linear_regression`](02_linear_regression.ipynb).